# Deep Learning Attribution Classification

This notebook demonstrates a binary classification workflow using **TensorFlow/Keras** to predict the binary `is_attributed` outcome from click-record identifiers and time-derived features.

The source notebook loads a large CSV from a local path containing the labels `kaggle-talkingdata2` and `competition_files`, but the notebook does not document enough provenance to identify a specific competition with certainty. This portfolio version therefore describes only the observed schema and modeling task. It modernizes the TensorFlow 1.x workflow with TensorFlow 2.x, explicit validation and test sets, class-imbalance handling, early stopping, and rare-event evaluation metrics.

> The source notebook predicts `is_attributed`. The precise meaning, collection process, and external origin of the dataset are not established within the notebook itself.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("TensorFlow version:", tf.__version__)

## 1. Load data

The source notebook loads `train.csv` and uses the following fields:

- `ip`
- `app`
- `device`
- `os`
- `channel`
- `click_time`
- `is_attributed`

The original source file is not distributed. When `train.csv` is unavailable, the notebook generates a synthetic sample using the same field names and binary target structure so the workflow remains reproducible. The synthetic data does not establish or reproduce the original dataset’s provenance.

In [ ]:
DATA_PATH = Path("train.csv")
MAX_ROWS = 500_000

def make_synthetic_attribution_data(n_rows=120_000, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)

    df = pd.DataFrame({
        "ip": rng.integers(1, 50_000, n_rows),
        "app": rng.integers(1, 350, n_rows),
        "device": rng.integers(1, 75, n_rows),
        "os": rng.integers(1, 200, n_rows),
        "channel": rng.integers(1, 500, n_rows),
        "click_time": pd.Timestamp("2017-11-06")
            + pd.to_timedelta(rng.integers(0, 72 * 3600, n_rows), unit="s"),
    })

    hour = df["click_time"].dt.hour.to_numpy()
    logit = (
        -6.6
        + 0.9 * np.isin(df["app"], [3, 12, 19, 35]).astype(float)
        + 0.7 * np.isin(df["channel"], [21, 111, 213, 347]).astype(float)
        + 0.45 * ((hour >= 0) & (hour < 5)).astype(float)
        + 0.25 * (df["device"] <= 4).astype(float)
    )
    probability = 1 / (1 + np.exp(-logit))
    df["is_attributed"] = rng.binomial(1, probability)
    return df

if DATA_PATH.exists():
    raw = pd.read_csv(DATA_PATH, nrows=MAX_ROWS)
    data_source = "local source file"
else:
    raw = make_synthetic_attribution_data()
    data_source = "synthetic fallback"

required_columns = {
    "ip", "app", "device", "os", "channel", "click_time", "is_attributed"
}
missing = required_columns - set(raw.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

print(f"Loaded {len(raw):,} rows from {data_source}.")
print(f"Positive rate: {raw['is_attributed'].mean():.3%}")
raw.head()

## 2. Feature engineering

High-cardinality identifiers are treated as numeric inputs in this compact historical demonstration. In a production system, embeddings, hashing, frequency encoding, or carefully validated categorical encoders would usually be considered.

In [ ]:
data = raw.copy()
data["click_time"] = pd.to_datetime(data["click_time"], errors="coerce")
data = data.dropna(subset=["click_time", "is_attributed"])

data["hour"] = data["click_time"].dt.hour
data["day_of_week"] = data["click_time"].dt.dayofweek
data["time_of_day"] = pd.cut(
    data["hour"],
    bins=[-1, 3, 7, 11, 15, 19, 23],
    labels=False,
)

feature_columns = [
    "ip", "app", "device", "os", "channel",
    "hour", "day_of_week", "time_of_day",
]
target_column = "is_attributed"

X = data[feature_columns].astype("float32")
y = data[target_column].astype("float32")

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE,
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype("float32")
X_valid_scaled = scaler.transform(X_valid).astype("float32")
X_test_scaled = scaler.transform(X_test).astype("float32")

print("Train:", X_train_scaled.shape)
print("Validation:", X_valid_scaled.shape)
print("Test:", X_test_scaled.shape)

## 3. Address class imbalance

In the source workflow, the positive `is_attributed` class is uncommon. Accuracy alone can therefore be misleading. The model uses class weights during training and is evaluated with ROC-AUC, PR-AUC, precision, recall, and a confusion matrix.

In [ ]:
negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())

if positive_count == 0:
    raise ValueError("The training sample contains no positive observations.")

class_weight = {
    0: 1.0,
    1: negative_count / positive_count,
}

print("Class weights:", class_weight)

## 4. Build the TensorFlow/Keras model

The network uses two dense hidden layers, batch normalization, dropout, and a sigmoid output for binary classification.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.20),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.AUC(name="roc_auc"),
        tf.keras.metrics.AUC(name="pr_auc", curve="PR"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
    ],
)

model.summary()

## 5. Train with early stopping

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_pr_auc",
        mode="max",
        patience=5,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-5,
    ),
]

history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_valid_scaled, y_valid),
    epochs=40,
    batch_size=1024,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)

## 6. Training history

In [ ]:
history_df = pd.DataFrame(history.history)

ax = history_df[["loss", "val_loss"]].plot(
    figsize=(8, 4),
    title="Training and validation loss",
)
ax.set_xlabel("Epoch")
ax.set_ylabel("Binary cross-entropy")
plt.show()

ax = history_df[["pr_auc", "val_pr_auc"]].plot(
    figsize=(8, 4),
    title="Training and validation PR-AUC",
)
ax.set_xlabel("Epoch")
ax.set_ylabel("PR-AUC")
plt.show()

## 7. Holdout evaluation

In [ ]:
test_probability = model.predict(X_test_scaled, verbose=0).ravel()

roc_auc = roc_auc_score(y_test, test_probability)
pr_auc = average_precision_score(y_test, test_probability)

print(f"Holdout ROC-AUC: {roc_auc:.4f}")
print(f"Holdout PR-AUC:  {pr_auc:.4f}")

In [ ]:
fpr, tpr, _ = roc_curve(y_test, test_probability)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("Holdout ROC curve")
plt.legend()
plt.show()

precision, recall, thresholds = precision_recall_curve(y_test, test_probability)
plt.figure(figsize=(6, 5))
plt.plot(recall, precision, label=f"PR-AUC = {pr_auc:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Holdout precision-recall curve")
plt.legend()
plt.show()

## 8. Threshold selection

The default 0.50 threshold is rarely optimal for a highly imbalanced use case. This example selects the threshold that maximizes the F1 score on the validation set and then applies that threshold once to the untouched test set.

In [ ]:
valid_probability = model.predict(X_valid_scaled, verbose=0).ravel()
valid_precision, valid_recall, valid_thresholds = precision_recall_curve(
    y_valid, valid_probability
)

f1 = (
    2 * valid_precision[:-1] * valid_recall[:-1]
    / np.maximum(valid_precision[:-1] + valid_recall[:-1], 1e-12)
)
best_index = int(np.nanargmax(f1))
selected_threshold = float(valid_thresholds[best_index])

print(f"Selected validation threshold: {selected_threshold:.4f}")
print(f"Validation F1: {f1[best_index]:.4f}")

In [ ]:
test_prediction = (test_probability >= selected_threshold).astype(int)

print(classification_report(y_test, test_prediction, digits=4))
print("Confusion matrix:")
print(confusion_matrix(y_test, test_prediction))

## 9. Save the model

The saved Keras artifact contains the trained network. The fitted scaler and feature definitions should also be versioned in a production implementation.

In [ ]:
MODEL_PATH = Path("attribution_classifier.keras")
model.save(MODEL_PATH)
print(f"Saved model to {MODEL_PATH.resolve()}")

## Limitations

- The original large source dataset is not distributed, and its exact external provenance is not verified from the notebook alone.
- The synthetic fallback reproduces selected field names and a rare binary outcome; it does not reproduce the original data-generating process or real-world performance.
- High-cardinality identifier fields receive simplified numeric treatment for compactness.
- Random splitting may overstate performance when temporal drift exists; a production evaluation should use time-based validation.
- Threshold selection should reflect business costs, campaign economics, and operational capacity.
- Predictive associations should not be interpreted as causal effects.